In [20]:
%pip install pandas numpy matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
#load dataset
file_path = 'C:/Users/Lish Ai Labs/Desktop/Simba/forex_data_M15_EUR_GBP.csv'
df = pd.read_csv(file_path)
print(df.head())


             timestamp data_source_id interval instrument     open     high  \
0  2025-02-01 00:00:00         MDSAgg      M15    EUR/GBP  0.83655  0.83685   
1  2025-01-31 23:45:00         MDSAgg      M15    EUR/GBP  0.83661  0.83668   
2  2025-01-31 23:30:00         MDSAgg      M15    EUR/GBP  0.83664  0.83676   
3  2025-01-31 23:15:00         MDSAgg      M15    EUR/GBP  0.83681  0.83681   
4  2025-01-31 23:00:00         MDSAgg      M15    EUR/GBP  0.83673  0.83689   

       low    close  
0  0.83547  0.83558  
1  0.83633  0.83655  
2  0.83660  0.83661  
3  0.83654  0.83664  
4  0.83665  0.83681  


In [9]:
# getting information about the dataset
print(df.info())       
print(df.describe())  
print(df.dtypes)  


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   timestamp       200 non-null    datetime64[ns]
 1   data_source_id  200 non-null    object        
 2   interval        200 non-null    object        
 3   instrument      200 non-null    object        
 4   open            200 non-null    float64       
 5   high            200 non-null    float64       
 6   low             200 non-null    float64       
 7   close           200 non-null    float64       
dtypes: datetime64[ns](1), float64(4), object(3)
memory usage: 12.6+ KB
None
                           timestamp        open        high         low  \
count                            200  200.000000  200.000000  200.000000   
mean   2025-01-30 23:07:29.999999744    0.836642    0.836812    0.836469   
min              2025-01-29 22:15:00    0.835450    0.835730    0.83541

In [6]:
# Convert 'timestamp' to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [7]:
# finding missing values
print(df.isnull().sum()) 

timestamp         0
data_source_id    0
interval          0
instrument        0
open              0
high              0
low               0
close             0
dtype: int64


In [ ]:
# Check for Implausible Values:
print("\nChecking for Implausible values:")
print(f"Number of rows with open price zero: {(df['open'] == 0).sum()}")
print(f"Number of rows with high price zero: {(df['high'] == 0).sum()}")
print(f"Number of rows with low price zero: {(df['low'] == 0).sum()}")
print(f"Number of rows with close price zero: {(df['close'] == 0).sum()}")

#Volume zero:
print(f"Number of rows with volume price zero: {(df['close'] == 0).sum()}")

print(f"Number of rows where high and low are the same: {(df['high'] == df['low']).sum()}")


Checking for Implausible values:
Number of rows with open price zero: 0
Number of rows with high price zero: 0
Number of rows with low price zero: 0
Number of rows with close price zero: 0
Number of rows with volume price zero: 0
Number of rows where high and low are the same: 0


In [ ]:
# check and remove duplicated timestamps:
print(f"Number of duplicated timestamps: {df.duplicated(subset=['timestamp']).sum()}") 
df.drop_duplicates(subset=['timestamp'], inplace=True) 

Number of duplicated timestamps: 0


In [18]:
# finding  and dealing with outliers
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

outlier_mask = pd.Series(False, index=df.index)

# Iterate over each numerical column and detect outliers
for col in numerical_cols:
    print(f"Processing column: {col}")  

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    
    col_outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)

    outlier_mask = outlier_mask | col_outlier_mask

    outlier_indices = col_outlier_mask[col_outlier_mask].index.tolist() 
    print(f"Outlier indices for column {col}: {outlier_indices}")

print("\nNumber of outliers per column:\n",  outlier_mask.value_counts())

# Remove rows containing outliers in ANY column
print("Number of outlier rows: ", outlier_mask.sum())  
df = df[~outlier_mask]  

print("Data with outlier rows: ", df.head())

Processing column: open
Outlier indices for column open: []
Processing column: high
Outlier indices for column high: []
Processing column: low
Outlier indices for column low: []
Processing column: close
Outlier indices for column close: []

Number of outliers per column:
 False    198
Name: count, dtype: int64
Number of outlier rows:  0
Data with outlier rows:              timestamp data_source_id interval instrument     open     high  \
0 2025-02-01 00:00:00         MDSAgg      M15    EUR/GBP  0.83655  0.83685   
1 2025-01-31 23:45:00         MDSAgg      M15    EUR/GBP  0.83661  0.83668   
2 2025-01-31 23:30:00         MDSAgg      M15    EUR/GBP  0.83664  0.83676   
3 2025-01-31 23:15:00         MDSAgg      M15    EUR/GBP  0.83681  0.83681   
4 2025-01-31 23:00:00         MDSAgg      M15    EUR/GBP  0.83673  0.83689   

       low    close  
0  0.83547  0.83558  
1  0.83633  0.83655  
2  0.83660  0.83661  
3  0.83654  0.83664  
4  0.83665  0.83681  


In [19]:
# finding any remaining outliers
print("\nall_outlier_indices:\n", all_outlier_indices)


all_outlier_indices:
       open   high    low  close
0    False  False  False  False
1    False  False  False  False
2    False  False  False  False
3    False  False  False  False
4    False  False  False  False
..     ...    ...    ...    ...
195  False  False  False  False
196  False  False  False  False
197  False  False  False  False
198  False  False  False  False
199  False  False  False  False

[200 rows x 4 columns]


In [21]:
# saving the cleaned dataset
 
folder_path = os.path.join(os.path.expanduser("~"), "Desktop", "Simba") 

file_name = "cleaned_forex_data.csv"
file_path = os.path.join(folder_path, file_name)

df.to_csv(file_path, index=False, encoding='utf-8')

print(f"Cleaned data saved to: {file_path}")

Cleaned data saved to: C:\Users\Lish Ai Labs\Desktop\Simba\cleaned_forex_data.csv
